In [1]:
import pandas as pd
import numpy as np
import os
import sys
import pickle
import random
import time
import itertools
from pathlib import Path
import tensorflow as tf

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

import matplotlib.pyplot as plt

from system import ATNNS_GPU

base_path = Path().resolve()

In [2]:
#Load example data and ensure correct formatting for model input
df = pd.read_parquet(base_path / 'data' / 'eaim_example_data.parquet')

df_actual = df[['phase', 'amm_nit', 'amm_chl', 'water_content']].copy()
df = df.drop(['water_content', 'p_H2O', 'p_HNO3', 'p_HCl', 'p_NH3', 'p_H2SO4', 'amm_nit', 'amm_chl', 'phase'], axis=1)

df_actual.reset_index(drop=True, inplace=True)
df.reset_index(drop=True, inplace=True)

df.head()

,TEMP,RH,NH4+,NA+,SO42-,NO3-,CL-
0,263.15,0.6,0.0,1.0,0.0,0.0,1.0
1,263.15,0.6,0.0,1.0,0.0,1.0,0.0
2,263.15,0.6,0.0,2.0,0.0,1.0,1.0
3,263.15,0.6,0.0,2.0,1.0,0.0,0.0
4,263.15,0.6,0.0,3.0,0.0,1.0,2.0


In [3]:
solver = ATNNS_GPU()

input_tensor = tf.convert_to_tensor(df.values, dtype=tf.float32)

results = solver.gpu_prediction(input_tensor)

In [4]:
final_results = pd.DataFrame(results.numpy())
final_results.columns = ['phase', 'amm_nit', 'amm_chl', 'water_content']
final_results.sample(10)

,phase,amm_nit,amm_chl,water_content
3259598,0.0,4.541059e-19,2.493718e-19,14556.194336
1660187,0.0,8.990258e-20,5.029071e-19,194.014786
3211941,0.0,3.903404e-15,1.614103e-15,29.923058
773699,0.0,6.101432e-21,3.683952e-20,335.155304
2831980,0.0,2.349683e-16,7.850689e-16,51.617344
2681923,0.0,5.566991e-20,6.008120e-20,9795.924805
3176744,0.0,8.720610e-20,6.272025e-19,12508.643555
1757797,0.0,9.534281e-18,8.521126e-17,51.884094
1918138,0.0,4.618563e-19,8.342114e-18,294.196960
1596568,0.0,1.573754e-18,3.081340e-18,128.102737


In [5]:
metrics = solver.evaluate_results(df_actual.drop('phase',axis=1), final_results.drop('phase', axis=1), inputs=df)
metrics

{'amm_nit': {'mape': 0.02322259872960621,
  'nmae': 0.10803434670351952,
  'rmse': 4.658663327037383e-16},
 'amm_chl': {'mape': 0.022516362606618014,
  'nmae': 0.06835283703254562,
  'rmse': 7.255653734989334e-16},
 'water_content': {'mass_error': 0.0430676816976375,
  'mape': 0.13195144250904178,
  'nmae': 0.025834529804049988,
  'rmse': 99.55711504592178}}

In [7]:
print(classification_report(df_actual['phase'], final_results['phase']))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00   3223483
           1       0.98      0.99      0.99    162099

    accuracy                           1.00   3385582
   macro avg       0.99      1.00      0.99   3385582
weighted avg       1.00      1.00      1.00   3385582

